# 13 — SAM 3 nativo desde HuggingFace *(Avanzado / Opcional)*

> Tener acceso aprobado al modelo de huggingface
> En HuggingFace y una sesión autenticada. Los notebooks del curso no dependen de este.

## ¿Qué vamos a construir hoy?

Usarás la API nativa de HuggingFace (`transformers`) para cargar SAM 3
directamente, sin el wrapper de Ultralytics. Luego convertirás el output
al formato de Supervision — demostrando que la librería funciona con
cualquier API, no solo con Ultralytics.

**Aprenderás a:**
- Cargar SAM 3 con `Sam3Processor` y `Sam3Model` de `transformers`
- Correr inferencia con prompts de texto y bounding boxes
- Convertir el output nativo a `sv.Detections` manualmente
- Entender qué abstrae Ultralytics y qué ganas con la API nativa

**Prerequisitos:**
- Cuenta en HuggingFace con acceso aprobado a `facebook/sam3`
- Token de acceso generado en https://huggingface.co/settings/tokens

## API nativa vs. wrapper de Ultralytics

En otros cuadernos usamos `SAM("sam3.pt")` de Ultralytics — conveniente pero limitado
a lo que el wrapper expone.

La API nativa de HuggingFace da acceso a:
- Todos los tipos de prompt: texto, cajas, puntos, máscaras, ejemplares visuales
- Control fino sobre umbrales de segmentación
- Inferencia en lote (batch) sobre múltiples imágenes
- Pesos del modelo sin conversión

```
Ultralytics:     SAM("sam3.pt")(image, bboxes=...)  ← simple
HuggingFace:     processor(images, text) → model() → post_process()  ← flexible
```

El costo: más código explícito. El beneficio: acceso completo al modelo.

Supervision sigue siendo el puente al final — el `sv.Detections` que
construyamos aquí es idéntico al de cualquier otro notebook.


In [ ]:
!pip install transformers torch supervision
!huggingface-cli login

import supervision as sv
from transformers import Sam3Processor, Sam3Model
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
import urllib.request
from pathlib import Path
from PIL import Image

Path("assets").mkdir(exist_ok=True)
urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg", "assets/bus.jpg")
urllib.request.urlretrieve("https://ultralytics.com/images/zidane.jpg", "assets/zidane.jpg")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")

In [ ]:
from huggingface_hub import login

# Enter your Hugging Face token securely when prompted.
login()


## Paso 1: Autenticarse y cargar el modelo

Si aún no has iniciado sesión en HuggingFace, ejecuta en la terminal:

```bash
huggingface-cli login
```

Pega tu token de acceso cuando se solicite. Solo necesitas hacerlo una vez por entorno.

Asegúrate también de haber solicitado y recibido acceso al modelo en:
https://huggingface.co/facebook/sam3


In [ ]:
processor = Sam3Processor.from_pretrained("facebook/sam3")
model     = Sam3Model.from_pretrained("facebook/sam3").to(device)
model.eval()

print("Modelo cargado")
print(f"Parámetros: {sum(p.numel() for p in model.parameters()) / 1e6:.0f} M")

## Paso 2: Inferencia con prompt de texto

La API de `transformers` tiene tres pasos:

1. **`processor()`** — prepara imagen y prompt como tensores
2. **`model()`** — ejecuta la red neuronal
3. **`post_process_instance_segmentation()`** — convierte el output bruto a máscaras y cajas

El resultado es un diccionario con `masks`, `boxes` y `scores`.
Necesitamos convertirlo a `sv.Detections` manualmente.

In [ ]:
def sam3_a_detections(results: dict) -> sv.Detections:
    masks  = results["masks"].cpu().numpy().astype(bool)
    xyxy   = results["boxes"].cpu().numpy()
    scores = results["scores"].cpu().numpy()
    return sv.Detections(xyxy=xyxy, mask=masks, confidence=scores)

image_pil = Image.open("assets/bus.jpg").convert("RGB")
image_bgr = cv2.imread("assets/bus.jpg")

inputs = processor(images=image_pil,text="person",return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs)
results = processor.post_process_instance_segmentation(outputs,threshold=0.5,mask_threshold=0.5,target_sizes=[image_pil.size[::-1]])[0]
detections = sam3_a_detections(results)
print(f"Objetos encontrados: {len(detections)}")
print(f"¿Tiene máscaras?    {detections.mask is not None}")
if detections.mask is not None:
    print(f"Shape de máscaras:  {detections.mask.shape}")

## Paso 3: Visualizar con Supervision

Una vez convertido a `sv.Detections`, el resto es idéntico a los cuadernos anteriores.

In [ ]:
mask_annotator  = sv.MaskAnnotator(opacity=0.6, color_lookup=sv.ColorLookup.INDEX)
label_annotator = sv.LabelAnnotator(text_scale=0.5, color_lookup=sv.ColorLookup.INDEX)
labels = [f"{c:.2f}" for c in detections.confidence]
annotated = mask_annotator.annotate(scene=image_bgr.copy(), detections=detections)
annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)
plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("SAM 3 nativo (transformers) → sv.Detections → MaskAnnotator")
plt.show()

## Pausa y observa: estructura del output nativo


In [ ]:
print("Claves del output post-procesado:", list(results.keys()))
print(f"masks  → tipo: {type(results['masks'])},  shape: {results['masks'].shape}")
print(f"boxes  → tipo: {type(results['boxes'])},  shape: {results['boxes'].shape}")
print(f"scores → tipo: {type(results['scores'])}, shape: {results['scores'].shape}")
print("Después de sam3_a_detections():")
print(f"  xyxy:       {detections.xyxy.shape}")
print(f"  mask:       {detections.mask.shape}")
print(f"  confidence: {detections.confidence}")

## 🔧 Exploración interactiva

### Experimento 1: Múltiples conceptos en un prompt


In [ ]:
inputs_multi = processor(images=image_pil,text=[["person", "vehicle"]],return_tensors="pt").to(device)
with torch.no_grad():
    outputs_multi = model(**inputs_multi)
results_multi = processor.post_process_instance_segmentation(outputs_multi,threshold=0.3,mask_threshold=0.3,target_sizes=[image_pil.size[::-1]])[0]
det_multi = sam3_a_detections(results_multi)
print(f"Objetos con ['person', 'vehicle']: {len(det_multi)}")
annotated_multi = sv.MaskAnnotator(opacity=0.6, color_lookup=sv.ColorLookup.INDEX).annotate(scene=image_bgr.copy(), detections=det_multi)
plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(annotated_multi, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Prompt: ['person', 'vehicle']")
plt.show()

In [ ]:
print(f"scores → tipo: {type(results_multi['scores'])}, shape: {results_multi['scores'].shape}")
print(f"Confidence: {det_multi.confidence}")

### Experimento 2: Texto vs. bounding box — ¿el mismo resultado?

Compara usar texto ("person") contra usar las cajas de YOLO.

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO
yolo_model = YOLO("yolov8n.pt")
yolo_r = yolo_model(image_bgr)[0]
yolo_det = sv.Detections.from_ultralytics(yolo_r)
print('yolo: ', len(yolo_det))
boxes_list = yolo_det.xyxy.tolist()
inputs_bbox = processor(images=image_pil,input_boxes=[boxes_list],input_boxes_labels=[[1] * len(boxes_list)],return_tensors="pt").to(device)
with torch.no_grad():
    outputs_bbox = model(**inputs_bbox)
results_bbox = processor.post_process_instance_segmentation(outputs_bbox,target_sizes=[image_pil.size[::-1]])[0]
det_bbox = sam3_a_detections(results_bbox)
scene_txt = sv.MaskAnnotator(opacity=0.6, color_lookup=sv.ColorLookup.INDEX).annotate(scene=image_bgr.copy(), detections=detections)
scene_bbox = sv.MaskAnnotator(opacity=0.6, color_lookup=sv.ColorLookup.INDEX).annotate(scene=image_bgr.copy(), detections=det_bbox)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
ax1.imshow(cv2.cvtColor(scene_txt, cv2.COLOR_BGR2RGB)); ax1.set_title('Prompt texto: "person"'); ax1.axis("off")
ax2.imshow(cv2.cvtColor(scene_bbox, cv2.COLOR_BGR2RGB)); ax2.set_title("Prompt bbox (YOLO)"); ax2.axis("off")
plt.tight_layout()
plt.show()

### Experimento 3: Efecto del umbral de confianza

`threshold` controla cuándo una detección se incluye en el resultado.
Un umbral bajo devuelve más objetos (incluidos falsos positivos); uno alto, solo los más seguros.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, thr in zip(axes, [0.2, 0.5, 0.8]):
    res_thr = processor.post_process_instance_segmentation(outputs,threshold=thr,mask_threshold=0.5,target_sizes=[image_pil.size[::-1]])[0]
    det_thr = sam3_a_detections(res_thr)
    scene = sv.MaskAnnotator(opacity=0.6, color_lookup=sv.ColorLookup.INDEX).annotate(scene=image_bgr.copy(), detections=det_thr)
    ax.imshow(cv2.cvtColor(scene, cv2.COLOR_BGR2RGB))
    ax.set_title(f"threshold={thr}  ({len(det_thr)} objetos)")
    ax.axis("off")
plt.tight_layout()
plt.show()